#### Memory Implementation

In [1]:
import json
import builtins
from crewai import Agent, Task, Crew, Process, LLM, Memory
from crewai.tools import BaseTool, tool
from pydantic import BaseModel,TypeAdapter, Field
from typing import List, Dict
from ai_module.config import create_memory

#
#memory = create_memory()

# 1. Define the individual object structure
class Item(BaseModel):
    id: int = Field(..., description="The id of the item")
    name: str = Field(..., description="The name of the item")
    price: float = Field(gt=0, description="The price of the item")
    quantity: int = Field(gt=0, description="the quantity of the item and quantity must be greater than zero")

# 2. Define the schema for the tool's arguments
class OrdersInput(BaseModel):
    records: List[Item] = Field(..., description="A list of items to process")

# Define a Pydantic model for the output structure
class AnalysisOutput(BaseModel):
    top_product: str
    total_orders: int
    total_quantity: int
    total_sell: float

# 3. Create the tool
class OrderAnalysisTool(BaseTool):
    name: str = "Order Analysis Tool"
    description: str = "Tool to perform order analysis."
    args_schema: type[BaseModel] = OrdersInput   # Link Pydantic here

    def _run(self, records: List[Item]) -> AnalysisOutput:
        # 4. Access the list of records directly
        t_sell = builtins.sum((r.quantity * r.price) for r in records)
        t_quantity = builtins.sum(r.quantity for r in records)
        top_product = builtins.max(records, key=lambda r: r.quantity)
        #
        # return f"Order Summary Report\n\n total_orders: {len(records)}\n total_sell: {t_sell}\n top_product: {top_product.name}\n total_quantity: {t_quantity}"
        return AnalysisOutput(
            top_product=top_product.name, 
            total_orders=len(records), 
            total_quantity=t_quantity, 
            total_sell=t_sell
        )

# Test OrderAnalysisTool Tool
# OrderAnalysisTool().run([
#     Item(id=1, name="Laptop", price=1500.00, quantity=4),
#     Item(id=2, name="Moniter", price=1600.00, quantity=8),
#     Item(id=3, name="CPU", price=250.00, quantity=16),
#     Item(id=4, name="RAM", price=150.00, quantity=12)
# ])   

# Create an agent to collect orders
order_collector = Agent(
    role='Order Collector',
    goal='Collect the list of orders from provided input order data.',
    backstory="You are an expert in processing input order data.",
    verbose=True,
    #max_iter=1,
    #memory=memory #.scope("/agent/order_collector"),
)

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a order summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    tools=[OrderAnalysisTool()],
    verbose=True,
    #max_iter=1,
    #memory=memory #.scope("/agent/order_analyst"),
)

# Define the task using the Pydantic models for input and output validation
order_collector_task = Task(
    description="Analyze the incoming list of orders {orders}.",
    expected_output="provide a pydentic object with  orders attribute that represent list of orders, " \
                    "make sure pydentic object include following attributes :"
                    "- orders",
    output_pydantic=OrdersInput,  # Pydantic model for output structure
    agent=order_collector
)

order_analysis_task = Task(
    description="Analyze the incoming orders from order_collector_task and provide a summary report.",
    expected_output="Analysis summary as pydentic object that should include 'total_orders', 'top_product', and 'total_quantity' fields in final output.",
    output_pydantic=AnalysisOutput,  # Pydantic model for output structure
    #tools=[OrderAnalysisTool()],
    context=[order_collector_task],
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[
        order_collector,
        order_analyst
    ],
    tasks=[
        order_collector_task,
        order_analysis_task
    ],
    process=Process.sequential,
    verbose=True,
    #memory=memory
)

#
from datetime import datetime
run_date = datetime.now().strftime('%Y-%m-%d')
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Sample input data (list of orders)
input_data = {
    "orders":[
        {"id": 1, "name":"Laptop", "price":150.00, "quantity": 4},
        {"id": 2, "name":"Monitor", "price":100.5, "quantity": 5},
        {"id": 3, "name":"Laptop", "price":150.00, "quantity": 3},
        {"id": 4, "name":"Monitor", "price":100.5, "quantity": 4}
    ],
    "run_date":run_date,
    "run_id":run_id
}

# Kick off the task
print(f"order analysis crew triggered on {run_date} with execution id {run_id}")
result = order_analysis_crew.kickoff(inputs=input_data)

# The result will be validated and structured by the output_model
print(result)

# Access the Pydantic object
# structured_data = processing_task.output.pydantic # returns a RecordList instance

# Iterate through your records
# for record in structured_data.items:
#     print(f"Processing: {record.name}")

/home/brijeshdhaker/IdeaProjects/bd-crewai-module/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


order analysis crew triggered on 2026-05-11 with execution id 20260511_081204


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b29934fe-bf27-458c-b82e-4bd6cd6cd659                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the incoming list of orders [{'id': 1, 'name': 'Laptop', 'price': 150.0, 'quantity': 4}, {'id':  │
│  2, 'name': 'Monitor', 'price': 100.5, 'quantity': 5}, {'id': 3, 'name': 'Laptop', 'price': 150.0, 'quantity':  │
│  3}, {'id': 4, 'name': 'Monitor', 'price': 100.5, 'quantity': 4}].                                              │
│  ID: c6d7439b-23c5-4dbd-920b-002c9f2bd1c1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Order Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  top_product='Laptop' total_orders=2 total_quantity=16 total_sell=1952.0                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the incoming orders from order_collector_task and provide a summary report.                      │
│  Agent: Order Analyst                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

top_product='Laptop' total_orders=2 total_quantity=16 total_sell=1952.0


╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 0c8e3909-e4a0-4af4-aa4a-034d5ea02a21                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/0c8e3909-e4a0-4af │
│ 4-aa4a-034d5ea02a21?access_code=TRACE-0bdd4e28c9                             │
│ 🔑 Access Code: TRACE-0bdd4e28c9                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b29934fe-bf27-458c-b82e-4bd6cd6cd659                                                                       │
│  Final Output: {"top_product":"Laptop","total_orders":2,"total_quantity":16,"total_sell":1952.0}                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
from crewai import Memory
from ai_module.config import create_memory

# https://knowledge.crewai.com/en/concepts/memory
memory = create_memory()

# Build up knowledge
memory.remember("The API rate limit is 1000 requests per minute.")
memory.remember("Our staging environment uses port 8080.")
memory.remember("The team agreed to use feature flags for all new releases.")

# Later, recall what you need
matches = memory.recall("What are our API limits?", limit=5)
for m in matches:
    print(f"[{m.score:.2f}] {m.record.content}")

# Extract atomic facts from a longer text
raw = """Meeting notes: We decided to migrate from MySQL to PostgreSQL
next quarter. The budget is $50k. Sarah will lead the migration."""

facts = memory.extract_memories(raw)
# ["Migration from MySQL to PostgreSQL planned for next quarter",
#  "Database migration budget is $50k",
#  "Sarah will lead the database migration"]

for fact in facts:
    memory.remember(fact)

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 73982.46ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 56467.25ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[0.69] The API rate limit is 1000 requests per minute.
[0.64] The team agreed to use feature flags for all new releases.
[0.63] Task: Analyze the incoming orders from order_collector_task and provide a summary report.
Agent: Order Analyst
Expected result: summary report should include 'total_orders', 'top_product', and 'total_quantity' fields in final report.
Result: {"total_orders": 4, "top_product": "Laptop", "total_quantity": 12}
[0.63] Our staging environment uses port 8080.
[0.63] Task: Analyze the incoming orders from order_collector_task and provide a summary report.
Agent: Order Analyst
Expected result: summary report should include 'total_orders', 'top_product', and 'total_quantity' fields in final report.
Result: {"total_orders":150,"top_product":"Laptop","total_quantity":27}


╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 68711.44ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from pydantic import BaseModel
from typing import List
from operator import attrgetter
import builtins

class User(BaseModel):
    name: str
    score: int

users = [
    User(name="Alice", score=85),
    User(name="Bob", score=95),
    User(name="Charlie", score=70)
]

# 1. Get the full User object with the highest score
top_user = builtins.max(users, key=attrgetter('score'))
print(top_user.name)  # Output: Bob

# 2. Get just the highest score value
highest_score = builtins.max(u.score for u in users)
print(highest_score)  # Output: 95
